# DiffuLLaMA POS primary-only adaptive completion

CPU-only fitting from the completed extraction. This protocol supports only the
paper's primary POS condition: progress `.50`, middle depth. It screens all 32
heads with seed 42, freezes top/bottom/uncertainty candidates, confirms them on
seeds 43/44, and publishes one matched middle-depth causal pair only after the
held-out stability gate passes.

Expected runtime: **3.5–4.2 hours**; hard cap: **5 hours**, including a 15-minute
validation reserve. Early/late and multi-progress POS claims are explicitly removed.
The GPU is disabled and no model is loaded.

In [ ]:
import json, os, subprocess
from pathlib import Path
from google.colab import drive

REPOSITORY = 'https://github.com/Dabsoysauce/latentrelationsondlm.git'
BRANCH = 'optimize/primary-pos-under-5h'
PROTOCOL_COMMIT = 'c5931fb775dc45d15c072fc7f95f5b289a52e33d'
RUN_ID = 'paper-restoration-v1-diffullama-pos-token-class-linear-probes'
RESULTS_ROOT = Path('/content/drive/MyDrive/dlmrel-paper-results (1)/diffullama')
RUN_DIR = RESULTS_ROOT / 'exploratory_extensions/diffullama_7b/ewt/pos_token_class_linear_probes' / RUN_ID
CHECKOUT = Path('/content/latentrelationsondlm-pos-primary')
LOCAL_CACHE = Path('/content/dlmrel-pos-primary-cache')
TOTAL_BUDGET_SECONDS = 5 * 60 * 60
VALIDATION_RESERVE_SECONDS = 15 * 60

def run(*args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=cwd, check=True, env=os.environ.copy())

os.environ.update({
    'CUDA_VISIBLE_DEVICES': '',
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1',
})
drive.mount('/content/drive')

In [ ]:
if not CHECKOUT.exists():
    run('git', 'clone', '--filter=blob:none', '--no-checkout', REPOSITORY, CHECKOUT)
run('git', 'fetch', 'origin', BRANCH, cwd=CHECKOUT)
run('git', 'checkout', '--detach', PROTOCOL_COMMIT, cwd=CHECKOUT)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=CHECKOUT, text=True).strip()
assert head == PROTOCOL_COMMIT, (head, PROTOCOL_COMMIT)
run('python', '-m', 'pip', 'install', '-q', '-e', '.[dev]', cwd=CHECKOUT)

In [ ]:
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Completed extraction run is missing: {RUN_DIR}')
feature_parquets = list((RUN_DIR / 'checkpoints').glob('paper-pos-*-features*.parquet'))
if len(feature_parquets) != 180:
    raise RuntimeError(
        f'Expected 180 extracted feature chunks; found {len(feature_parquets)}. '
        'Do not start fitting or load a model.'
    )
logical_cpus = os.cpu_count() or 1
worker_counts = [n for n in (6, 8, 10, 12) if n <= logical_cpus] or [max(1, logical_cpus)]
print(json.dumps({
    'run_dir': str(RUN_DIR),
    'feature_parquets': len(feature_parquets),
    'logical_cpus': logical_cpus,
    'workers_to_benchmark': worker_counts,
}, indent=2))
run(
    'python', '-m', 'pytest', '-q',
    'tests/test_paper_pos_primary_adaptive.py',
    'tests/test_paper_pos_adaptive.py',
    'tests/test_paper_pos_adaptive_12.py',
    'tests/test_paper_optimizations.py',
    cwd=CHECKOUT,
)

In [ ]:
run(
    'python', '-m', 'dlmrel.cli', 'pos-fit-primary-adaptive',
    '--run-dir', RUN_DIR,
    '--local-cache', LOCAL_CACHE,
    '--resume',
    '--budget-seconds', str(TOTAL_BUDGET_SECONDS),
    '--validation-reserve-seconds', str(VALIDATION_RESERVE_SECONDS),
    '--worker-counts', *map(str, worker_counts),
    cwd=CHECKOUT,
)

In [ ]:
run(
    'python', '-m', 'dlmrel.cli', 'validate-pos-primary-adaptive',
    '--run-dir', RUN_DIR,
    cwd=CHECKOUT,
)
OUTPUT = RUN_DIR / 'pos_adaptive_primary'
manifest = json.loads((OUTPUT / 'adaptive_manifest.json').read_text())
assert manifest['status'] == 'confirmed'
print(json.dumps({
    'status': manifest['status'],
    'primary_condition_only': manifest['primary_condition_only'],
    'elapsed_hours': manifest['elapsed_seconds'] / 3600,
    'full_workers': manifest['full_workers'],
    'main_workers': manifest['main_workers'],
    'choices': str(OUTPUT / 'pos_head_choices.csv'),
    'rankings': str(OUTPUT / 'pos_head_rankings_primary.csv'),
    'coverage': str(OUTPUT / 'coverage_manifest.csv'),
}, indent=2))